# 🛸 Noise Texture Features & Physics-Informed Quick Wins Experiments

This notebook provides an ordered, end-to-end experimental workflow for:
1. **Task 2**: Clean Re-Baseline Benchmark (10-feature baseline on clean telemetry)
2. **Task 3**: Sensor Noise Texture Engineering (`prediction_error_autocorrelation`, `position_residual_std`, `speed_spectral_entropy`)
3. **Task 4**: Noise Texture 13-Feature Pipeline Evaluation (targeting the Geometry attack blindspot)
4. **Task 5**: Quick Wins — Deterministic Physics Rules & PE Distribution Feature Ensembling
5. **Comparative Analysis**: Synthesis across baseline, noise texture, and physics-rule augmented pipelines.

## 1. Setup Environment & Working Directory

In [ ]:
from pathlib import Path
import os
import sys
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Operating Directory: {os.getcwd()}")
print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")

## 2. TASK 2: Clean Re-Baseline Experiment (10 Features)

Runs the 10-feature baseline across pointwise classical models and deep sequence autoencoders.
Filtered clean dataset excludes simulated normal/hard flights.

In [ ]:
!python presets/run_gpu_pipeline.py \
    --preset baseline_10 \
    --exp-name exp5_baseline_10_clean \
    --model-family all \
    --epochs 15 \
    --patience 7 \
    --no-model-cache

## 3. TASK 3 & 4: Noise Texture Feature Pipeline (13 Features)

Evaluates the **noise texture hypothesis**: Real DJI GPS telemetry contains correlated multipath noise and sensor jitter, while simulated trajectories are mathematically smooth.

Features added:
- `prediction_error_autocorrelation`: Lag-1 autocorrelation of rolling prediction error.
- `position_residual_std`: Savitzky-Golay detrended position noise standard deviation.
- `speed_spectral_entropy`: Spectral entropy of ground speed fluctuations.

In [ ]:
!python presets/run_gpu_pipeline.py \
    --preset noise_texture_13 \
    --exp-name exp6_noise_texture_13 \
    --model-family all \
    --epochs 15 \
    --patience 7 \
    --no-model-cache

## 4. TASK 5: Quick Wins — Deterministic Physics Rules & Ensembles

Applies hard aerodynamic and kinematic threshold checks as a deterministic ensemble layer:
1. **Sustained Prediction Error**: `prediction_error > 10.0m` sustained for 3+ consecutive samples.
2. **Heading-Speed Inconsistency**: `heading_speed_consistency > 90 deg` at speed > 2.0 m/s sustained for 2+ samples.
3. **Yaw Acceleration Violation**: `abs(yaw_acceleration) > 500.0 deg/s^2`.

In [ ]:
from presets.run_task5_quick_wins import run_physics_ensemble_evaluation

task5_results_df = run_physics_ensemble_evaluation(feature_preset="noise_texture_13", threshold_sigma=3.0)
display(task5_results_df)

## 5. TASK 5 (Option B): Extended PE Distribution Features (16 Features)

Evaluates adding window-level prediction error distribution statistics (`pe_window_mean`, `pe_window_var`, `pe_window_skew`).

In [ ]:
!python presets/run_gpu_pipeline.py \
    --preset task5_extended_16 \
    --exp-name exp7_task5_extended_16 \
    --model-family pointwise \
    --no-model-cache

## 6. End-to-End Comparative Synthesis & Visualization

In [ ]:
from implement.utils.helper import get_output_dir

exp5_path = get_output_dir() / "gpu_experiments" / "exp5_baseline_10_clean" / "exp5_baseline_10_clean_results.csv"
exp6_path = get_output_dir() / "gpu_experiments" / "exp6_noise_texture_13" / "exp6_noise_texture_13_results.csv"
task5_path = get_output_dir() / "task5_quick_wins" / "task5_evaluation_noise_texture_13.csv"

print("=== EXPERIMENT COMPARISON SUMMARY ===")
if exp5_path.exists():
    print("\n--- Task 2 Baseline (10 Features) ---")
    display(pd.read_csv(exp5_path))
if exp6_path.exists():
    print("\n--- Task 4 Noise Texture (13 Features) ---")
    display(pd.read_csv(exp6_path))
if task5_path.exists():
    print("\n--- Task 5 Physics Rule Ensemble ---")
    display(pd.read_csv(task5_path))